# w1 · Tabular models (KKBox churn)

*Four comparable non-neural tabular models (M1 logistic, M2 LightGBM, M3 XGBoost,
M4 RandomForest), per-family Optuna search, one shared split, common metrics,
confusion matrices, final refit and reloadable scoring.*

The KPI is **log-loss** (probability quality). The real churn rate (~9%) is kept:
no SMOTE, no subsampling of classes, no manual rescaling of probabilities.

## Catalogue

| ID | Visible name | Input | Optimisation |
|----|--------------|-------|--------------|
| M1 | Logistic regression | Table | own Optuna |
| M2 | LightGBM | Table | own Optuna |
| M3 | XGBoost | Table | own Optuna |
| M4 | RandomForest | Table | own Optuna |

## One evaluation, holdout kept out of every decision

- **holdout**: the 20% of users whose membership expires latest (ordered by
  `days_to_expire`, msno hash as a tie-break). Saved before any training. This is an
  **expiry holdout**, not cross-month validation: every labelled user is in the same
  competition cutoff.
- **development** (remaining 80%) is split once by a stable msno hash, stratified by
  label, into `fit` (~75%), `stop` (~12.5%) and `tune` (~12.5%).

Optuna trains on `fit`, early-stops on `stop`, and reports log-loss on `tune`.
Thresholds and the winner are fixed on development **before** holdout is touched. Then
each family is refit on all development and evaluated once on holdout. Finally the winner
is refit on 100% of train for inference. Test has no labels.

The neural models (3 LSTM + hybrid MLP) live in the w2 neural notebook.


## 1. Configuration

`SMOKE_TEST=True` runs a tiny sample with few trials to validate the code; its output goes
to a separate folder and is clearly marked as a test. The default (`SMOKE_TEST=False`) uses
every eligible user. Smoke results are never published as FULL.


In [ ]:
from pathlib import Path
import gc
import json
import hashlib
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

SEED = 110

# --- Run mode -------------------------------------------------------------
# A smoke test exercises every code path with a small sample and few trials.
# Keep it True to validate quickly; set it to False for the full run.
SMOKE_TEST = False

# At most 50 Optuna trials TOTAL across families, not 50 per model.
# This is a search budget, not a guarantee of the global optimum.
# Refit training is separate from this budget.
OPTUNA_MAX_TRIALS = 50
TRIALS_FULL = {"M1": 3, "M2": 8, "M3": 8, "M4": 6}
assert sum(TRIALS_FULL.values()) <= OPTUNA_MAX_TRIALS
TRIALS_SMOKE = {k: 2 for k in TRIALS_FULL}
TRIALS = TRIALS_SMOKE if SMOKE_TEST else TRIALS_FULL
TRAIN_VERBOSE = 1  # Logistic iteration progress.
TREE_LOG_PERIOD = 100  # Boosting progress within each Optuna trial.

# Tree budgets.
TREE_MAX_ROUNDS = 50 if SMOKE_TEST else 3000
TREE_EARLY_STOP = 20 if SMOKE_TEST else 100
# Smoke sample size (users) taken from each phase; None = use all users.
SMOKE_USERS = 10 if SMOKE_TEST else None
# Smoke also limits test rows scored at the end (None = all test users).
SMOKE_TEST_ROWS = 10 if SMOKE_TEST else None

print("SMOKE_TEST:", SMOKE_TEST, "| trials:", TRIALS,
      "| total trials:", sum(TRIALS.values()), "| tree rounds:", TREE_MAX_ROUNDS)


In [ ]:
# --- Paths ----------------------------------------------------------------
# Local tabular export from the w1 preprocessing pipeline; on Kaggle it is uploaded.
ON_KAGGLE = Path("/kaggle/input").exists()
INPUT_DIR = Path("/kaggle/input/dataset-final") if ON_KAGGLE else Path("export-tabular-v3")
_base_out = Path("/kaggle/working") if ON_KAGGLE else Path("artifacts")
OUT_DIR = _base_out / ("sherlock_w1_smoke" if SMOKE_TEST else "sherlock_w1")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("INPUT_DIR:", INPUT_DIR, "| OUT_DIR:", OUT_DIR, "| ON_KAGGLE:", ON_KAGGLE)


### Local path override (optional)

**Kaggle:** leave the next cell commented; the configuration above is correct.

**Local:** uncomment the lines to read `export-sequential-v3/` from the project folder.
These paths replace the ones above. Comment them again to return to Kaggle.

In [ ]:
# LOCAL DATA: uncomment to use the local export.
# ON_KAGGLE = False
# INPUT_DIR = Path("export-tabular-v3")
# OUT_DIR = (Path("artifacts") / ("sherlock_w1_smoke" if SMOKE_TEST else "sherlock_w1"))
# OUT_DIR.mkdir(parents=True, exist_ok=True)
# print("LOCAL override ->", INPUT_DIR, OUT_DIR)


In [ ]:
# --- Library availability -------------------------------------------------
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    log_loss, roc_auc_score, average_precision_score, brier_score_loss,
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, confusion_matrix, roc_curve,
    precision_recall_curve, classification_report,
)
from sklearn.calibration import calibration_curve

import lightgbm as lgb
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
import xgboost as xgb
from xgboost import XGBClassifier
import optuna
import joblib

optuna.logging.set_verbosity(optuna.logging.WARNING)

import random
random.seed(SEED)
np.random.seed(SEED)

# XGBoost needs a CUDA build; otherwise it runs on CPU and says so.
try:
    _xgb_cuda = bool(xgb.build_info().get("USE_CUDA", False))
except Exception:
    _xgb_cuda = False
XGB_DEVICE = "cuda" if _xgb_cuda else "cpu"


def _lgbm_gpu_available():
    probe_x = np.random.RandomState(SEED).rand(400, 4)
    probe_y = (probe_x[:, 0] > 0.5).astype(int)
    try:
        LGBMClassifier(device_type="gpu", n_estimators=2, verbose=-1).fit(probe_x, probe_y)
        return True
    except Exception as exc:
        print("LightGBM GPU probe failed:", repr(exc), flush=True)
        return False


LGBM_DEVICE = ({"device_type": "gpu", "max_bin": 63, "gpu_use_dp": False}
               if _lgbm_gpu_available() else {})

ENV = {
    "numpy": np.__version__, "pandas": pd.__version__,
    "sklearn": __import__("sklearn").__version__, "lightgbm": lgb.__version__,
    "xgboost": xgb.__version__, "optuna": optuna.__version__,
    "xgb_device": XGB_DEVICE, "lgbm_device": LGBM_DEVICE.get("device_type", "cpu"),
}
print("Environment:", json.dumps(ENV, indent=2))
if XGB_DEVICE == "cpu":
    print("NOTE: XGBoost running on CPU (no CUDA build).")
if not LGBM_DEVICE:
    print("NOTE: LightGBM running on CPU (GPU kernel unavailable).")


## 2. Tabular v3 contract and loading

Validate `dataset_version == 3` before reading anything. Read the feature catalogue and the
categorical columns from `metadata.json`, then confirm the parquet `msno` order reproduces
the published row hash. We keep an explicit original-row index aligned with the tables.


In [ ]:
META = json.loads((INPUT_DIR / "metadata.json").read_text())
if META.get("dataset_version") != 3:
    raise ValueError(f"Expected dataset_version 3, got {META.get('dataset_version')!r}.")
if META["cutoffs"] != {"train": "2017-02-28", "test": "2017-03-31"}:
    raise ValueError(f"Unexpected cutoffs: {META['cutoffs']}")

FEATURES = META["features"]
CATEGORICAL = META["categorical"]
NUMERIC = [c for c in FEATURES if c not in CATEGORICAL]
assert not {"msno", "is_churn"}.intersection(FEATURES)

print("dataset_version:", META["dataset_version"],
      "| features:", len(FEATURES), "| categorical:", CATEGORICAL)


In [ ]:
# Read tables; keep an explicit original row index aligned with the tensors.
train_df = pd.read_parquet(INPUT_DIR / "train.parquet",
                           columns=["msno", "is_churn"] + FEATURES)
test_df = pd.read_parquet(INPUT_DIR / "test.parquet", columns=["msno"] + FEATURES)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
train_df["row"] = np.arange(len(train_df))
test_df["row"] = np.arange(len(test_df))

assert train_df["msno"].is_unique and test_df["msno"].is_unique
y_all = train_df["is_churn"].to_numpy(dtype=np.int32)
assert set(np.unique(y_all)) == {0, 1}

# Confirm the parquet msno order reproduces the published row hash.
def _order_hash(msno_series):
    digest = hashlib.sha1()
    for value in msno_series:
        digest.update(str(value).encode()); digest.update(b"\n")
    return digest.hexdigest()

assert _order_hash(train_df["msno"]) == META["hash_train"], "train parquet misaligned"
assert _order_hash(test_df["msno"]) == META["hash_test"], "test parquet misaligned"

print("train:", train_df.shape, "| test:", test_df.shape,
      "| churn rate:", round(float(y_all.mean()), 4))

## 3. Single shared split (fit / stop / tune / holdout)

One protocol for every model. The **holdout** is the 20% of users with the latest
`days_to_expire` (msno hash breaks ties deterministically), saved before any training.
The remaining 80% is **development**, split once by a stable msno hash stratified by
label into `fit` (~75% of development), `stop` (~12.5%) and `tune` (~12.5%).

These are internal parts of a single evaluation, not alternative experiments. Every
model uses the same users in each part. We validate disjointness, coverage, both
classes and order, and save sizes/prevalences in one diagnostic.

In [ ]:
def _msno_hash_unit(msno):
    """Stable hash in [0, 1) from msno, independent of RNG call order."""
    h = hashlib.sha1(("salt110:" + str(msno)).encode()).hexdigest()
    return int(h[:15], 16) / float(16 ** 15)

hash_unit = train_df["msno"].map(_msno_hash_unit).to_numpy()

# Holdout: 20% latest expiry, msno hash as the deterministic tie-break.
order_key = np.lexsort((hash_unit, train_df["days_to_expire"].to_numpy()))
n_holdout = int(round(len(train_df) * 0.20))
holdout_rows = np.sort(order_key[-n_holdout:])
dev_rows = np.sort(order_key[:-n_holdout])

# Development split by stable hash, stratified by label (no repeated RNG draws).
# Thresholds on the per-class hash rank give exact, reproducible stratified shares.
def _stratified_hash_assign(rows, shares):
    """Assign each row to a part using its per-class hash rank. shares sum to 1."""
    assign = np.empty(len(rows), dtype=object)
    names = list(shares)
    cum = np.cumsum([shares[n] for n in names])
    for cls in (0, 1):
        cls_rows = rows[y_all[rows] == cls]
        ranks = np.argsort(np.argsort(hash_unit[cls_rows])) / max(len(cls_rows), 1)
        part_idx = np.searchsorted(cum, ranks, side="right")
        part_idx = np.clip(part_idx, 0, len(names) - 1)
        assign[np.searchsorted(rows, cls_rows)] = np.asarray(names, dtype=object)[part_idx]
    return assign

dev_shares = {"fit": 0.75, "stop": 0.125, "tune": 0.125}
dev_assign = _stratified_hash_assign(dev_rows, dev_shares)
SPLIT = {
    "fit": np.sort(dev_rows[dev_assign == "fit"]),
    "stop": np.sort(dev_rows[dev_assign == "stop"]),
    "tune": np.sort(dev_rows[dev_assign == "tune"]),
    "holdout": holdout_rows,
}
DEV_ROWS = dev_rows  # all development rows (fit+stop+tune), for the development refit

In [ ]:
# Optional smoke subsample: shrink every part but keep both classes and disjointness.
if SMOKE_USERS is not None:
    rng = np.random.default_rng(SEED)
    sized = {part: SMOKE_USERS for part in SPLIT}
    small = {}
    for part, idx in SPLIT.items():
        take = min(sized[part], len(idx))
        pos = idx[y_all[idx] == 1]
        neg = idx[y_all[idx] == 0]
        n_pos = max(1, int(round(take * len(pos) / len(idx))))
        n_pos = min(n_pos, len(pos)); n_neg = min(take - n_pos, len(neg))
        chosen = np.concatenate([rng.choice(pos, n_pos, replace=False),
                                 rng.choice(neg, n_neg, replace=False)])
        small[part] = np.sort(chosen)
    SPLIT = small
    DEV_ROWS = np.sort(np.concatenate([SPLIT["fit"], SPLIT["stop"], SPLIT["tune"]]))
    print("SMOKE split sizes:", {k: len(v) for k, v in SPLIT.items()})

In [ ]:
# Validate: disjoint, both classes, sorted; save indices and a single diagnostic.
_seen = {}
for part, idx in SPLIT.items():
    assert np.all(np.diff(idx) > 0), f"{part} not sorted/unique"
    assert set(np.unique(y_all[idx])) == {0, 1}, f"{part} missing a class"
    for other, oidx in _seen.items():
        assert not np.intersect1d(idx, oidx).size, f"{part} overlaps {other}"
    _seen[part] = idx

split_summary = pd.DataFrame([
    {"part": part, "n_users": len(idx),
     "churn_rate": round(float(y_all[idx].mean()), 5),
     "n_churn": int(y_all[idx].sum()), "n_stay": int((y_all[idx] == 0).sum())}
    for part, idx in SPLIT.items()
])
np.savez_compressed(OUT_DIR / "split_indices.npz",
                    **{k: v for k, v in SPLIT.items()}, development=DEV_ROWS)
split_summary.to_csv(OUT_DIR / "split_summary.csv", index=False)
FIT_RATE = float(y_all[SPLIT["fit"]].mean())   # training prevalence for M0 and biases
print("saved split_indices.npz and split_summary.csv | fit churn rate:", round(FIT_RATE, 5))
split_summary

## 4. Preprocessors and design matrices

The tabular preprocessor imputes, applies signed log1p, clips and standardises the numeric
columns, and one-hot encodes the categorical columns. It is fit **only** on each phase's
training users (`fit`, all development for the development refit, or all train for the final
refit); categories are never unioned with test. The tree models use native categoricals or
an ordinal fit-vocabulary (RandomForest), never the test levels.


In [ ]:
def numeric_values(frame, columns):
    values = frame[columns].to_numpy(dtype=np.float64, copy=True)
    values[~np.isfinite(values)] = np.nan
    return values


def category_values(series):
    return series.astype("string").fillna("__MISSING__").to_numpy(dtype=str)


def fit_tabular_preprocessor(frame, features, categorical):
    """Median impute, signed log1p, p0.5/p99.5 clip, standardize; one-hot with fit levels."""
    numeric = [c for c in features if c not in categorical]
    raw = numeric_values(frame, numeric)
    median = np.array([np.nanmedian(col) if np.isfinite(col).any() else 0.0 for col in raw.T])
    x = np.where(np.isnan(raw), median, raw)
    x = np.sign(x) * np.log1p(np.abs(x))
    lower, upper = np.quantile(x, [0.005, 0.995], axis=0)
    x = np.clip(x, lower, upper)
    mean, std = x.mean(axis=0), x.std(axis=0)
    std[std < 1e-6] = 1.0
    levels = {c: sorted(set(category_values(frame[c]))) for c in categorical}
    return {"numeric": numeric, "categorical": list(categorical),
            "median": median.tolist(), "lower": lower.tolist(), "upper": upper.tolist(),
            "mean": mean.tolist(), "std": std.tolist(), "levels": levels}


def transform_tabular(frame, state):
    raw = numeric_values(frame, state["numeric"])
    x = np.where(np.isnan(raw), np.asarray(state["median"]), raw)
    x = np.sign(x) * np.log1p(np.abs(x))
    x = np.clip(x, state["lower"], state["upper"])
    pieces = [(x - np.asarray(state["mean"])) / np.asarray(state["std"])]
    for col in state["categorical"]:
        values = category_values(frame[col])
        pieces.append(np.column_stack([values == level for level in state["levels"][col]]))
    out = np.concatenate(pieces, axis=1).astype(np.float32)
    assert np.isfinite(out).all()
    return out

In [ ]:
# ---- Phase scalers: fit once per training-row set, reuse across trials ----
# A "phase" names the rows a scaler is fit on. We fit on fit (for Optuna / development
# search), on development (for the development refit) and on all train (final refit).
PHASE_ROWS = {"fit": SPLIT["fit"], "development": DEV_ROWS,
              "train": np.arange(len(train_df))}
if SMOKE_USERS is not None:
    PHASE_ROWS["train"] = np.sort(np.concatenate(list(SPLIT.values())))

_scaler_cache = {}

def phase_scalers(phase):
    """Return the tabular preprocessor for a phase."""
    if phase not in _scaler_cache:
        rows = PHASE_ROWS[phase]
        tab = fit_tabular_preprocessor(train_df.iloc[rows], FEATURES, CATEGORICAL)
        _scaler_cache[phase] = tab
    return _scaler_cache[phase]


In [ ]:
# ---- Transformed batch access by original row index -----------------------
def tabular_batch(rows, phase, dataset="train"):
    tab = phase_scalers(phase)
    frame = (train_df if dataset == "train" else test_df).iloc[np.asarray(rows)]
    return transform_tabular(frame, tab)


## 5. Metrics and confusion-matrix helpers

One evaluation function takes labels, probabilities, a model id and an **already-fixed**
threshold; it never trains or optimises anything. The threshold is chosen by maximum F1
of churn on **tune** with a deterministic tie rule (prefer the higher threshold). Both
classes are required for AUC; `zero_division=0` and empty denominators are handled.

In [ ]:
EPS = 1e-7

def select_threshold(y_true, proba):
    """Threshold maximising churn F1; ties prefer the higher threshold (deterministic)."""
    y_true = np.asarray(y_true); proba = np.asarray(proba)
    grid = np.linspace(0.01, 0.99, 99)
    best_t, best_f1 = 0.5, -1.0
    for t in grid:
        f1 = f1_score(y_true, (proba >= t).astype(int), zero_division=0)
        if f1 >= best_f1:            # >= so ties keep the higher threshold
            best_f1, best_t = f1, float(t)
    return best_t


def evaluate(y_true, proba, model_id, threshold):
    """Full metric row at a fixed threshold. No training, no threshold search here."""
    y_true = np.asarray(y_true).astype(int)
    proba = np.clip(np.asarray(proba, dtype=np.float64), EPS, 1 - EPS)
    pred = (proba >= threshold).astype(int)
    two_classes = len(np.unique(y_true)) == 2
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    return {
        "model_id": model_id,
        "log_loss": log_loss(y_true, proba, labels=[0, 1]),
        "roc_auc": roc_auc_score(y_true, proba) if two_classes else np.nan,
        "average_precision": average_precision_score(y_true, proba) if two_classes else np.nan,
        "brier": brier_score_loss(y_true, proba),
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "precision_churn": precision_score(y_true, pred, zero_division=0),
        "recall_churn": recall_score(y_true, pred, zero_division=0),
        "specificity": specificity,
        "f1_churn": f1_score(y_true, pred, zero_division=0),
        "f1_macro": f1_score(y_true, pred, average="macro", zero_division=0),
        "mcc": matthews_corrcoef(y_true, pred) if two_classes else 0.0,
        "threshold": float(threshold),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "support_0": int((y_true == 0).sum()), "support_1": int((y_true == 1).sum()),
        "actual_churn_rate": float(y_true.mean()),
        "predicted_churn_rate": float(pred.mean()),
    }


def metrics_at_half(y_true, proba, model_id):
    return evaluate(y_true, proba, model_id, 0.5)

In [ ]:
# ---- Confusion-matrix figures (counts + row-normalised) -------------------
CLASS_LABELS = ["Permanece (0)", "Churn (1)"]

def _draw_confusion(ax_counts, ax_norm, y_true, proba, threshold, title):
    y_true = np.asarray(y_true).astype(int)
    pred = (np.asarray(proba) >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    row_sums = cm.sum(axis=1, keepdims=True)
    norm = np.divide(cm, row_sums, out=np.zeros_like(cm, dtype=float),
                     where=row_sums != 0) * 100.0
    # Counts panel (own colour scale).
    im0 = ax_counts.imshow(cm, cmap="Blues")
    ax_counts.figure.colorbar(im0, ax=ax_counts, fraction=0.046, pad=0.04)
    names = ["TN", "FP", "FN", "TP"]
    for (r, c), v in np.ndenumerate(cm):
        ax_counts.text(c, r, f"{names[r*2+c]}\n{v:d}", ha="center", va="center",
                       color="white" if v > cm.max() / 2 else "black", fontsize=10)
    ax_counts.set_title(f"{title}\ncounts @ thr={threshold:.2f}", fontsize=10)
    # Row-normalised panel (shared 0-100 scale).
    im1 = ax_norm.imshow(norm, cmap="Blues", vmin=0, vmax=100)
    ax_norm.figure.colorbar(im1, ax=ax_norm, fraction=0.046, pad=0.04)
    for (r, c), v in np.ndenumerate(norm):
        ax_norm.text(c, r, f"{v:.1f}%", ha="center", va="center",
                     color="white" if v > 50 else "black", fontsize=10)
    ax_norm.set_title(f"{title}\nrow-normalised %", fontsize=10)
    for ax in (ax_counts, ax_norm):
        ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
        ax.set_xticklabels(CLASS_LABELS, rotation=20, ha="right", fontsize=8)
        ax.set_yticklabels(CLASS_LABELS, fontsize=8)
        ax.set_xlabel("Predicción"); ax.set_ylabel("Clase real")


def save_confusion(y_true, proba, threshold, model_id, out_dir, tag="holdout"):
    """Save a single high-resolution confusion figure (counts + normalised) as PNG+SVG."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    _draw_confusion(axes[0], axes[1], y_true, proba, threshold, f"{model_id} ({tag})")
    fig.tight_layout()
    fig.savefig(out_dir / f"confusion_{model_id}_{tag}.png", dpi=200, bbox_inches="tight")
    fig.savefig(out_dir / f"confusion_{model_id}_{tag}.svg", bbox_inches="tight")
    return fig


def gallery_confusion(entries, thresholds, out_dir, tag="holdout", per_page=3):
    """Paginated gallery, up to `per_page` models per figure (counts + normalised cols)."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    items = list(entries.items())
    figs = []
    for page in range(0, len(items), per_page):
        chunk = items[page:page + per_page]
        fig, axes = plt.subplots(len(chunk), 2, figsize=(9, 4 * len(chunk)), squeeze=False)
        for r, (model_id, (y_true, proba)) in enumerate(chunk):
            _draw_confusion(axes[r, 0], axes[r, 1], y_true, proba,
                            thresholds[model_id], f"{model_id} ({tag})")
        fig.tight_layout()
        fig.savefig(out_dir / f"gallery_{tag}_{page // per_page}.png", dpi=160, bbox_inches="tight")
        figs.append(fig)
    return figs

In [ ]:
def save_classification_report(y_true, proba, threshold, model_id, out_dir):
    """Full classification_report (per class + macro/weighted) as JSON and text."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    pred = (np.asarray(proba) >= threshold).astype(int)
    rep_dict = classification_report(y_true, pred, labels=[0, 1],
                                     target_names=CLASS_LABELS, output_dict=True,
                                     zero_division=0)
    rep_text = classification_report(y_true, pred, labels=[0, 1],
                                     target_names=CLASS_LABELS, zero_division=0)
    with (out_dir / f"report_{model_id}.json").open("w") as f:
        json.dump(rep_dict, f, indent=2)
    (out_dir / f"report_{model_id}.txt").write_text(rep_text)
    return rep_dict

## 6. Model registry

A registry stores, per model id, how to fit, predict, save and load a predictor, plus its
inputs, best hyper-parameters, trial accounting, timings and the chosen threshold. A load
function reconstructs each predictor without relying on notebook global state. Blends
reference their frozen components and weights.

In [ ]:
# Everything a model produced lives here; nothing downstream reads training globals.
REGISTRY = {}   # model_id -> dict(record)
PROBA = {}      # model_id -> {"tune": .., "holdout": .., "test": ..} probability arrays

def register(model_id, **record):
    REGISTRY[model_id] = {"model_id": model_id, **record}
    return REGISTRY[model_id]


def set_proba(model_id, part, proba):
    PROBA.setdefault(model_id, {})[part] = np.clip(np.asarray(proba, dtype=np.float64),
                                                   EPS, 1 - EPS)


def rows_of(part):
    return SPLIT[part]


def y_of(part):
    return y_all[SPLIT[part]]

In [ ]:
# ---- Optuna study helpers (SQLite persistence, resumable budgets) ---------
OPTUNA_DIR = OUT_DIR / "optuna"
OPTUNA_DIR.mkdir(parents=True, exist_ok=True)
STORAGE = f"sqlite:///{(OPTUNA_DIR / 'studies.sqlite3').as_posix()}"

# A fingerprint of the data contract + split; a study is not resumed if it changes.
CONTRACT = {
    "dataset_version": META["dataset_version"],
    "hash_train": META["hash_train"], "hash_test": META["hash_test"],
    "split_sizes": {k: int(len(v)) for k, v in SPLIT.items()},
    "smoke": SMOKE_TEST,
    "libs": {k: ENV[k] for k in ("lightgbm", "xgboost", "optuna", "sklearn")},
}
CONTRACT_HASH = hashlib.sha1(json.dumps(CONTRACT, sort_keys=True).encode()).hexdigest()[:12]


def get_study(model_id):
    """Create or resume a persistent study; reject a different data/split contract."""
    name = f"{model_id}_{CONTRACT_HASH}"
    if model_id == "M1":
        name += "_newton_cholesky_v1"  # Do not resume SAGA trials with another solver.
    study = optuna.create_study(
        study_name=name, storage=STORAGE, load_if_exists=True,
        direction="minimize", sampler=optuna.samplers.TPESampler(
            seed=SEED, n_startup_trials=max(1, min(3, TRIALS[model_id] // 2))),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=1),
    )
    study.set_user_attr("contract", CONTRACT)
    return study


def remaining_trials(study, total):
    """Count all attempted trials, including failed/interrupted ones, against budget."""
    done = sum(t.state != optuna.trial.TrialState.WAITING for t in study.trials)
    return max(0, total - done)


def run_study(model_id, objective, total_trials, enqueue=None):
    study = get_study(model_id)
    if enqueue:
        existing = {json.dumps(t.params, sort_keys=True) for t in study.trials}
        for params in enqueue:
            if json.dumps(params, sort_keys=True) not in existing:
                study.enqueue_trial(params, skip_if_exists=True)
    n = remaining_trials(study, total_trials)
    print(f"{model_id}: budget={total_trials}, remaining={n}", flush=True)

    def objective_with_progress(trial):
        print(f"{model_id}: starting trial #{trial.number}", flush=True)
        return objective(trial)

    def report_trial(study, trial):
        seconds = trial.duration.total_seconds() if trial.duration else 0.0
        print(f"{model_id}: trial #{trial.number} {trial.state.name} "
              f"| log-loss={trial.value} | {seconds:.1f}s", flush=True)

    if n > 0:
        study.optimize(objective_with_progress, n_trials=n, gc_after_trial=True,
                       show_progress_bar=True, callbacks=[report_trial])
    if not any(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials):
        raise RuntimeError(f"{model_id}: study has no completed trial; cannot select params.")
    df = study.trials_dataframe()
    df.to_csv(OPTUNA_DIR / f"{model_id}_trials.csv", index=False)
    with (OPTUNA_DIR / f"{model_id}_best_params.json").open("w") as f:
        json.dump(study.best_params, f, indent=2)
    counts = {
        "complete": sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials),
        "pruned": sum(t.state == optuna.trial.TrialState.PRUNED for t in study.trials),
        "failed": sum(t.state == optuna.trial.TrialState.FAIL for t in study.trials),
    }
    if counts["complete"] == 0:
        raise RuntimeError(f"{model_id}: study has no completed trial; cannot select params.")
    print(f"{model_id}: best tune log-loss={study.best_value:.5f} | trials {counts}")
    return study, counts

## 7. Tabular studies (M1, M2, M3, M4)

M1 (logistic), M2 (LightGBM), M3 (XGBoost) and M4 (RandomForest) each get their own Optuna
study that trains on `fit`, early-stops on `stop` (boosting models) and reports log-loss on
`tune`. Preprocessors are fit only on the phase's training users; categories are never
unioned with test. M1 uses `newton-cholesky`, suited to the many-rows/few-features binary
case here. M4 encodes categoricals ordinally using the fit vocabulary.


In [ ]:
# Development-phase tabular design inputs, fit only on `fit`.
# For the logistic pipeline we build an sklearn ColumnTransformer fit on `fit` rows.
y_fit, y_stop, y_tune = y_of("fit"), y_of("stop"), y_of("tune")

def _frame(part):
    return train_df.iloc[rows_of(part)]


In [ ]:
# ---- M1: logistic regression (own Optuna) -------------------------------
def _make_logistic_pre():
    return ColumnTransformer([
        ("num", StandardScaler(), NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), CATEGORICAL),
    ])

def _fit_logistic(C, rows_train):
    pipe = Pipeline([("pre", _make_logistic_pre()),
                     ("clf", LogisticRegression(C=C, penalty="l2",
                                                solver="newton-cholesky", max_iter=200,
                                                random_state=SEED, verbose=TRAIN_VERBOSE))])
    print(f"M1: newton-cholesky C={C:.6g} on {len(rows_train):,} rows (CPU)", flush=True)
    with warnings.catch_warnings():
        warnings.simplefilter("always")  # do not hide convergence warnings
        pipe.fit(train_df.iloc[rows_train][FEATURES], y_all[rows_train])
    print("M1: iterations:", pipe.named_steps["clf"].n_iter_.tolist(), flush=True)
    return pipe

def M1_objective(trial):
    C = trial.suggest_float("C", 1e-4, 1e2, log=True)
    pipe = _fit_logistic(C, rows_of("fit"))
    p_tune = pipe.predict_proba(_frame("tune")[FEATURES])[:, 1]
    return log_loss(y_tune, np.clip(p_tune, EPS, 1 - EPS), labels=[0, 1])

def run_M1():
    t0 = time.perf_counter()
    study, counts = run_study("M1", M1_objective, TRIALS["M1"],
                              enqueue=[{"C": 1.0}])
    best_C = study.best_params["C"]
    model = _fit_logistic(best_C, rows_of("fit"))   # refit best on fit for dev predictions
    set_proba("M1", "tune", model.predict_proba(_frame("tune")[FEATURES])[:, 1])
    set_proba("M1", "holdout", model.predict_proba(_frame("holdout")[FEATURES])[:, 1])
    register("M1", family="logistic", input="table", n_channels="N/A",
             n_parameters="N/A", best_params=study.best_params, trials=counts,
             fit_seconds=time.perf_counter() - t0, estimator=model, best_C=best_C)
    print("M1 best C:", best_C)

run_M1()

In [ ]:
# ---- M2: LightGBM (own Optuna, native categoricals, early stopping) ------
# LightGBM uses pandas category dtype for native handling; set categories from fit.
# Tree categories use the vocabulary of the phase's training users only;
# unseen levels in other parts/test become missing (both libraries handle NaN).
_tree_vocab = {}

def tree_vocab(phase):
    if phase not in _tree_vocab:
        rows = PHASE_ROWS[phase]
        _tree_vocab[phase] = {c: sorted(pd.unique(train_df[c].iloc[rows].dropna()).tolist())
                              for c in CATEGORICAL}
    return _tree_vocab[phase]

def tree_frame(rows, phase, dataset="train", vocab=None):
    frame = (train_df if dataset == "train" else test_df).iloc[np.asarray(rows)][FEATURES].copy()
    vocab = vocab or tree_vocab(phase)
    for col in CATEGORICAL:
        frame[col] = pd.Categorical(frame[col], categories=vocab[col])
    return frame

def _lgbm_frame(rows):
    return tree_frame(rows, "fit")

X_fit_lgb = _lgbm_frame(rows_of("fit"))
X_stop_lgb = _lgbm_frame(rows_of("stop"))
X_tune_lgb = _lgbm_frame(rows_of("tune"))

def _fit_lgbm(params, X_tr, y_tr, X_es, y_es):
    model = LGBMClassifier(objective="binary", n_estimators=TREE_MAX_ROUNDS,
                           random_state=SEED, n_jobs=-1, verbose=-1,
                           **params, **LGBM_DEVICE)
    model.fit(X_tr, y_tr, eval_set=[(X_es, y_es)], eval_metric="binary_logloss",
              categorical_feature=CATEGORICAL,
              callbacks=[early_stopping(TREE_EARLY_STOP, verbose=True),
                         log_evaluation(TREE_LOG_PERIOD)])
    return model

def M2_objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 127),
        "max_depth": trial.suggest_categorical("max_depth", [-1, 4, 6, 8, 10, 12]),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 300),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "subsample_freq": 1,   # make bagging effective for subsample
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
    }
    model = _fit_lgbm(params, X_fit_lgb, y_fit, X_stop_lgb, y_stop)
    p_tune = model.predict_proba(X_tune_lgb)[:, 1]
    trial.set_user_attr("best_iteration", int(model.best_iteration_ or TREE_MAX_ROUNDS))
    return log_loss(y_tune, np.clip(p_tune, EPS, 1 - EPS), labels=[0, 1])

def run_M2():
    t0 = time.perf_counter()
    study, counts = run_study("M2", M2_objective, TRIALS["M2"], enqueue=[{
        "learning_rate": 0.05, "num_leaves": 63, "max_depth": -1,
        "min_child_samples": 100, "subsample": 0.9, "colsample_bytree": 0.9,
        "reg_alpha": 1e-4, "reg_lambda": 1.0}])
    best = dict(study.best_params); best["subsample_freq"] = 1
    model = _fit_lgbm(best, X_fit_lgb, y_fit, X_stop_lgb, y_stop)
    set_proba("M2", "tune", model.predict_proba(X_tune_lgb)[:, 1])
    set_proba("M2", "holdout", model.predict_proba(_lgbm_frame(rows_of("holdout")))[:, 1])
    register("M2", family="lightgbm", input="table", n_channels="N/A",
             n_parameters="N/A", best_params=study.best_params, trials=counts,
             fit_seconds=time.perf_counter() - t0, estimator=model,
             best_iteration=int(model.best_iteration_ or TREE_MAX_ROUNDS))
    print("M2 best iteration:", model.best_iteration_)

run_M2()

In [ ]:
# ---- M3: XGBoost (own Optuna, explicit categorical support) --------------
def _xgb_frame(rows):
    return tree_frame(rows, "fit")

X_fit_xgb = _xgb_frame(rows_of("fit"))
X_stop_xgb = _xgb_frame(rows_of("stop"))
X_tune_xgb = _xgb_frame(rows_of("tune"))

def _fit_xgb(params, X_tr, y_tr, X_es, y_es):
    model = XGBClassifier(
        n_estimators=TREE_MAX_ROUNDS, tree_method="hist", device=XGB_DEVICE,
        enable_categorical=True, eval_metric="logloss",
        early_stopping_rounds=TREE_EARLY_STOP, random_state=SEED, n_jobs=-1, **params)
    model.fit(X_tr, y_tr, eval_set=[(X_es, y_es)], verbose=TREE_LOG_PERIOD)
    return model

def M3_objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 30.0),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "gamma": trial.suggest_float("gamma", 1e-4, 5.0, log=True),
    }
    model = _fit_xgb(params, X_fit_xgb, y_fit, X_stop_xgb, y_stop)
    p_tune = model.predict_proba(X_tune_xgb)[:, 1]
    trial.set_user_attr("best_iteration", int(getattr(model, "best_iteration", TREE_MAX_ROUNDS)))
    return log_loss(y_tune, np.clip(p_tune, EPS, 1 - EPS), labels=[0, 1])

def run_M3():
    t0 = time.perf_counter()
    study, counts = run_study("M3", M3_objective, TRIALS["M3"], enqueue=[{
        "learning_rate": 0.05, "max_depth": 6, "min_child_weight": 1.0,
        "subsample": 0.9, "colsample_bytree": 0.9, "reg_alpha": 1e-4,
        "reg_lambda": 1.0, "gamma": 1e-4}])
    model = _fit_xgb(study.best_params, X_fit_xgb, y_fit, X_stop_xgb, y_stop)
    set_proba("M3", "tune", model.predict_proba(X_tune_xgb)[:, 1])
    set_proba("M3", "holdout", model.predict_proba(_xgb_frame(rows_of("holdout")))[:, 1])
    register("M3", family="xgboost", input="table", n_channels="N/A",
             n_parameters="N/A", best_params=study.best_params, trials=counts,
             fit_seconds=time.perf_counter() - t0, estimator=model,
             best_iteration=int(getattr(model, "best_iteration", TREE_MAX_ROUNDS)))
    print("M3 best iteration:", getattr(model, "best_iteration", None))

run_M3()

In [ ]:
# ---- M4: RandomForest (own Optuna, ordinal categories) -------------------
_rf_vocab = {}

def rf_vocab(phase):
    if phase not in _rf_vocab:
        rows = PHASE_ROWS[phase]
        _rf_vocab[phase] = {c: sorted(pd.unique(train_df[c].iloc[rows].dropna()).tolist())
                            for c in CATEGORICAL}
    return _rf_vocab[phase]

def encode_rf_frame(frame, vocab):
    """Ordinal codes from a fit vocabulary; unseen levels become -1."""
    out = frame[FEATURES].copy()
    for col in CATEGORICAL:
        codes = {level: i for i, level in enumerate(vocab[col])}
        out[col] = out[col].map(codes).fillna(-1).astype("float64")
    return out

def rf_frame(rows, phase, dataset="train", vocab=None):
    frame = (train_df if dataset == "train" else test_df).iloc[np.asarray(rows)]
    return encode_rf_frame(frame, vocab or rf_vocab(phase))

def _fit_rf(params, rows_train, phase="fit"):
    model = RandomForestClassifier(random_state=SEED, n_jobs=-1, **params)
    model.fit(rf_frame(rows_train, phase), y_all[rows_train])
    return model

def M4_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 800, step=100),
        "max_depth": trial.suggest_categorical("max_depth", [None, 6, 8, 10, 12, 16]),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 50),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 50),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.5, 1.0]),
    }
    model = _fit_rf(params, rows_of("fit"))
    p_tune = model.predict_proba(rf_frame(rows_of("tune"), "fit"))[:, 1]
    return log_loss(y_tune, np.clip(p_tune, EPS, 1 - EPS), labels=[0, 1])

def run_M4():
    t0 = time.perf_counter()
    study, counts = run_study("M4", M4_objective, TRIALS["M4"], enqueue=[{
        "n_estimators": 400, "max_depth": None, "min_samples_leaf": 5,
        "min_samples_split": 2, "max_features": "sqrt"}])
    best = dict(study.best_params)
    model = _fit_rf(best, rows_of("fit"))
    set_proba("M4", "tune", model.predict_proba(rf_frame(rows_of("tune"), "fit"))[:, 1])
    set_proba("M4", "holdout", model.predict_proba(rf_frame(rows_of("holdout"), "fit"))[:, 1])
    register("M4", family="randomforest", input="table", n_channels="N/A",
             n_parameters="N/A", best_params=best, trials=counts,
             fit_seconds=time.perf_counter() - t0, estimator=model)
    print("M4 best params:", best)

run_M4()


## 10. Thresholds, development refit and the single holdout evaluation

Thresholds come from maximum churn F1 on **tune** (deterministic tie rule). The winner is
chosen by tune log-loss and written to a manifest **before** holdout is touched. Each
trainable family is refit on all development (preprocessors refit on development only) and
evaluated once on holdout. Thresholds are not re-searched on holdout; measuring how the tune
threshold behaves after refit is part of the honest report.


In [ ]:
# ---- Thresholds and winner selection on tune (before holdout) ------------
MODEL_IDS = ["M1", "M2", "M3", "M4"]
THRESHOLDS = {}
for mid in MODEL_IDS:
    THRESHOLDS[mid] = select_threshold(y_tune, PROBA[mid]["tune"])

tune_losses = {mid: log_loss(y_tune, PROBA[mid]["tune"], labels=[0, 1]) for mid in MODEL_IDS}
primary_winner = min(MODEL_IDS, key=lambda m: tune_losses[m])

SELECTION = {
    "contract_hash": CONTRACT_HASH,
    "tune_log_loss": {m: round(float(v), 6) for m, v in tune_losses.items()},
    "thresholds": {m: round(float(t), 4) for m, t in THRESHOLDS.items()},
    "primary_winner": primary_winner,
}
with (OUT_DIR / "selection_manifest.json").open("w") as f:
    json.dump(SELECTION, f, indent=2)
print("primary winner (tune):", primary_winner)
print("thresholds:", {m: round(t, 3) for m, t in THRESHOLDS.items()})


In [ ]:
# ---- Development refit: retrain each family on all development ------------
# Preprocessors use the development phase; no holdout early stop.
DEV = "development"
y_dev = y_all[DEV_ROWS]
hold_rows = rows_of("holdout")
y_hold = y_all[hold_rows]

def _dev_lgbm_frame(rows):
    return tree_frame(rows, "development")

HOLD_PROBA = {}   # model_id -> holdout probability from the development-refit model

# M1 logistic on all development
_m1 = _fit_logistic(REGISTRY["M1"]["best_C"], DEV_ROWS)
HOLD_PROBA["M1"] = _m1.predict_proba(train_df.iloc[hold_rows][FEATURES])[:, 1]
REGISTRY["M1"]["dev_estimator"] = _m1

# M2 LightGBM: refit on development with fixed params and the tuned best_iteration
_m2_params = dict(REGISTRY["M2"]["best_params"]); _m2_params["subsample_freq"] = 1
_m2 = LGBMClassifier(objective="binary", n_estimators=REGISTRY["M2"]["best_iteration"],
                     random_state=SEED, n_jobs=-1, verbose=-1, **_m2_params, **LGBM_DEVICE)
_m2.fit(_dev_lgbm_frame(DEV_ROWS), y_dev, categorical_feature=CATEGORICAL)
HOLD_PROBA["M2"] = _m2.predict_proba(_dev_lgbm_frame(hold_rows))[:, 1]
REGISTRY["M2"]["dev_estimator"] = _m2

# M3 XGBoost: refit on development, fixed params and best_iteration
_m3 = XGBClassifier(n_estimators=max(1, REGISTRY["M3"]["best_iteration"]),
                    tree_method="hist", device=XGB_DEVICE, enable_categorical=True,
                    eval_metric="logloss", random_state=SEED, n_jobs=-1,
                    **REGISTRY["M3"]["best_params"])
_m3.fit(_dev_lgbm_frame(DEV_ROWS), y_dev, verbose=False)
HOLD_PROBA["M3"] = _m3.predict_proba(_dev_lgbm_frame(hold_rows))[:, 1]
REGISTRY["M3"]["dev_estimator"] = _m3

# M4 RandomForest on all development
_m4 = _fit_rf(REGISTRY["M4"]["best_params"], DEV_ROWS, phase="development")
HOLD_PROBA["M4"] = _m4.predict_proba(rf_frame(hold_rows, "development"))[:, 1]
REGISTRY["M4"]["dev_estimator"] = _m4

for mid in MODEL_IDS:
    set_proba(mid, "holdout", HOLD_PROBA[mid])
print("holdout probabilities ready for all tabular models")


## 11. Metrics tables, confusion matrices and curves

Holdout metrics for all tabular models at their tune threshold, plus a complementary table
at threshold 0.5. Confusion matrices (counts + row-normalised) per model and as a paginated
gallery. ROC/PR curves, reliability diagrams, Optuna histories and tree importances. Order
the display table by holdout log-loss for inspection, distinct from the winner fixed earlier
on tune.


In [ ]:
# ---- Holdout metrics tables (tune threshold and 0.5) ----------------------
FIG_DIR = OUT_DIR / "figures"
(FIG_DIR / "confusion").mkdir(parents=True, exist_ok=True)
(FIG_DIR / "curves").mkdir(parents=True, exist_ok=True)
REPORT_DIR = OUT_DIR / "metrics" / "classification_reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

rows_main, rows_half = [], []
for mid in MODEL_IDS:
    rec = REGISTRY[mid]
    row = evaluate(y_hold, PROBA[mid]["holdout"], mid, THRESHOLDS[mid])
    row.update({
        "family": rec["family"], "input": rec["input"],
        "n_channels": rec.get("n_channels", "N/A"),
        "n_parameters": rec.get("n_parameters", "N/A"),
        "trials_complete": rec["trials"]["complete"],
        "trials_pruned": rec["trials"]["pruned"],
        "trials_failed": rec["trials"]["failed"],
        "fit_seconds": round(float(rec.get("fit_seconds", 0.0)), 2),
        "best_epoch": rec.get("best_epoch", "N/A"),
        "best_iteration": rec.get("best_iteration", "N/A"),
        "weight": rec.get("weight", "N/A"),
        "correction_kept": rec.get("sequence_correction_kept_dev",
                                   rec.get("sequence_correction_kept", "N/A")),
        "best_params": json.dumps(rec.get("best_params", "N/A")),
    })
    rows_main.append(row)
    rows_half.append(metrics_at_half(y_hold, PROBA[mid]["holdout"], mid))
    save_classification_report(y_hold, PROBA[mid]["holdout"], THRESHOLDS[mid], mid, REPORT_DIR)

holdout_metrics = pd.DataFrame(rows_main).sort_values("log_loss").reset_index(drop=True)
holdout_metrics_at_05 = pd.DataFrame(rows_half).sort_values("log_loss").reset_index(drop=True)
holdout_metrics.to_csv(OUT_DIR / "metrics" / "holdout_metrics.csv", index=False)
holdout_metrics_at_05.to_csv(OUT_DIR / "metrics" / "holdout_metrics_at_05.csv", index=False)

_display_cols = ["model_id", "family", "log_loss", "roc_auc", "average_precision",
                 "brier", "accuracy", "balanced_accuracy", "precision_churn",
                 "recall_churn", "specificity", "f1_churn", "f1_macro", "mcc", "threshold"]
print("Holdout metrics ordered by log-loss (winner fixed on tune:", primary_winner, ")")
holdout_metrics[_display_cols].round(5)

In [ ]:
# ---- Confusion matrices: per-model images + paginated gallery -------------
hold_entries = {mid: (y_hold, PROBA[mid]["holdout"]) for mid in MODEL_IDS}
for mid in MODEL_IDS:
    fig = save_confusion(y_hold, PROBA[mid]["holdout"], THRESHOLDS[mid], mid,
                         FIG_DIR / "confusion", tag="holdout")
    plt.close(fig)
_gallery = gallery_confusion(hold_entries, THRESHOLDS, FIG_DIR / "confusion",
                             tag="holdout", per_page=3)
for f in _gallery:
    plt.show()
print("saved per-model confusion images (PNG+SVG) and gallery pages")

In [ ]:
# ---- ROC and precision-recall curves --------------------------------------
fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(14, 6))
for mid in MODEL_IDS:
    p = PROBA[mid]["holdout"]
    fpr, tpr, _ = roc_curve(y_hold, p)
    ax_roc.plot(fpr, tpr, label=f"{mid} (AUC={roc_auc_score(y_hold, p):.3f})")
    prec, rec, _ = precision_recall_curve(y_hold, p)
    ax_pr.plot(rec, prec, label=f"{mid} (AP={average_precision_score(y_hold, p):.3f})")
ax_roc.plot([0, 1], [0, 1], "k--", alpha=0.5, label="chance")
ax_roc.set(title="ROC (holdout)", xlabel="FPR", ylabel="TPR"); ax_roc.legend(fontsize=7)
ax_pr.axhline(float(y_hold.mean()), color="k", ls="--", alpha=0.5, label="prevalence")
ax_pr.set(title="Precision-Recall (holdout)", xlabel="Recall", ylabel="Precision")
ax_pr.legend(fontsize=7)
fig.tight_layout()
fig.savefig(FIG_DIR / "curves" / "roc_pr.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# ---- Reliability diagrams and probability histograms (diagnostics only) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for mid in MODEL_IDS:
    p = PROBA[mid]["holdout"]
    frac_pos, mean_pred = calibration_curve(y_hold, p, n_bins=10, strategy="quantile")
    axes[0].plot(mean_pred, frac_pos, marker="o", label=mid, alpha=0.7)
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[0].set(title="Reliability (holdout, diagnostic only)",
            xlabel="Mean predicted probability", ylabel="Observed churn rate")
axes[0].legend(fontsize=7)
best_p = PROBA[primary_winner]["holdout"]
axes[1].hist(best_p[y_hold == 0], bins=40, alpha=0.6, label="Permanece (0)", density=True)
axes[1].hist(best_p[y_hold == 1], bins=40, alpha=0.6, label="Churn (1)", density=True)
axes[1].set(title=f"Probability histogram — {primary_winner}", xlabel="probability")
axes[1].legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "curves" / "calibration_hist.png", dpi=160, bbox_inches="tight")
plt.show()
print("NOTE: reliability curves are diagnostics; no probability calibration was applied.")

In [ ]:
# ---- Optuna optimisation history per family with enough completed trials ---
for mid in ["M1", "M2", "M3", "M4"]:
    try:
        study = get_study(mid)
    except Exception:
        continue
    complete = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    if len(complete) >= 3:
        ax = optuna.visualization.matplotlib.plot_optimization_history(study)
        ax.set_title(f"Optuna history — {mid}")
        plt.show()
    else:
        print(f"{mid}: {len(complete)} completed trials — history plot skipped (need >= 3).")


In [ ]:
# ---- Tree feature importance (diagnostic) ---------------------------------
booster = REGISTRY["M2"].get("dev_estimator") or REGISTRY["M2"]["estimator"]
importance = pd.Series(booster.booster_.feature_importance(importance_type="gain"),
                       index=FEATURES).sort_values(ascending=False)
top_share = importance.iloc[0] / max(importance.sum(), 1)
print("Top feature:", importance.index[0], "| share of total gain:", round(float(top_share), 3))
if top_share > 0.5:
    print("WARNING: one feature dominates the gain — audit it for leakage.")
importance.head(20).plot(kind="barh", figsize=(8, 7))
plt.gca().invert_yaxis()
plt.title("M2 LightGBM feature importance (gain, top 20) — diagnostic")
plt.xlabel("gain"); plt.tight_layout()
plt.savefig(FIG_DIR / "curves" / "lgbm_importance.png", dpi=160, bbox_inches="tight")
plt.show()

## 12. Final refit on 100% of train and reloadable scoring

Decisions are frozen before holdout. Now retrain M1, M2, M3 and M4 on all train with the
selected params/budgets. Preprocessors are refit on all train only here. Test has no labels:
it never produces evaluation metrics. We score test in blocks, restore the original msno
order, and save one submission per model plus `submission.csv` (the winner).

A load function (`load_predictor`) rebuilds each predictor from its manifest without notebook
globals. `score_model(model_id, ...)` validates inputs, joins by unique msno and uses the
saved threshold; it never silently falls back to another model.


In [ ]:
# Final-phase preprocessors (fit on all train) and the final design matrices.
TRAIN_ROWS = PHASE_ROWS["train"]
y_train_all = y_all[TRAIN_ROWS]
FINAL_RATE = float(y_train_all.mean())

MODELS_DIR = OUT_DIR / "models" / "final"
PRE_DIR = OUT_DIR / "preprocessing" / "final"
PRED_DIR = OUT_DIR / "predictions"
SUB_DIR = OUT_DIR / "submissions"
for d in (MODELS_DIR, PRE_DIR, PRED_DIR, SUB_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Persist the final preprocessor (fit on all train).
_tab_f = phase_scalers("train")
with (PRE_DIR / "tabular_preprocess.json").open("w") as f:
    json.dump(_tab_f, f, indent=2)

FINAL_MODELS = {}   # model_id -> fitted final estimator
TEST_PROBA = {}
test_all_rows = np.arange(len(test_df) if SMOKE_TEST_ROWS is None else SMOKE_TEST_ROWS)
scored_test = test_df.iloc[test_all_rows]


In [ ]:
# ---- Final tabular models (M1, M2, M3) on all train ----------------------
def _final_lgbm_frame(rows, dataset="train"):
    return tree_frame(rows, "train", dataset)

# M1
m1_final = _fit_logistic(REGISTRY["M1"]["best_C"], TRAIN_ROWS)
TEST_PROBA["M1"] = m1_final.predict_proba(scored_test[FEATURES])[:, 1]
FINAL_MODELS["M1"] = m1_final
import joblib
joblib.dump(m1_final, MODELS_DIR / "M1_logistic.joblib")

# M2
_m2p = dict(REGISTRY["M2"]["best_params"]); _m2p["subsample_freq"] = 1
m2_final = LGBMClassifier(objective="binary", n_estimators=REGISTRY["M2"]["best_iteration"],
                          random_state=SEED, n_jobs=-1, verbose=-1, **_m2p, **LGBM_DEVICE)
m2_final.fit(_final_lgbm_frame(TRAIN_ROWS), y_train_all, categorical_feature=CATEGORICAL)
TEST_PROBA["M2"] = m2_final.predict_proba(_final_lgbm_frame(test_all_rows, "test"))[:, 1]
FINAL_MODELS["M2"] = m2_final
m2_final.booster_.save_model(str(MODELS_DIR / "M2_lightgbm.txt"))

# M3
m3_final = XGBClassifier(n_estimators=max(1, REGISTRY["M3"]["best_iteration"]),
                         tree_method="hist", device=XGB_DEVICE, enable_categorical=True,
                         eval_metric="logloss", random_state=SEED, n_jobs=-1,
                         **REGISTRY["M3"]["best_params"])
m3_final.fit(_final_lgbm_frame(TRAIN_ROWS), y_train_all, verbose=False)
TEST_PROBA["M3"] = m3_final.predict_proba(_final_lgbm_frame(test_all_rows, "test"))[:, 1]
FINAL_MODELS["M3"] = m3_final
m3_final.save_model(str(MODELS_DIR / "M3_xgboost.json"))
print("final tabular models trained on all train")

# M4 RandomForest final on all train
m4_final = _fit_rf(REGISTRY["M4"]["best_params"], TRAIN_ROWS, phase="train")
TEST_PROBA["M4"] = m4_final.predict_proba(rf_frame(test_all_rows, "train", dataset="test"))[:, 1]
FINAL_MODELS["M4"] = m4_final
joblib.dump(m4_final, MODELS_DIR / "M4_randomforest.joblib")
print("final RandomForest trained on all train")

In [ ]:
# ---- Final predictions validation and submissions -------------------------
# Validate ranges/length/finiteness and restore original msno order (already original).
for mid in MODEL_IDS:
    p = np.asarray(TEST_PROBA[mid], dtype=np.float64)
    assert p.shape == (len(scored_test),), f"{mid} wrong length"
    assert np.isfinite(p).all() and (p >= 0).all() and (p <= 1).all(), f"{mid} invalid probs"
    pd.DataFrame({"msno": scored_test["msno"].to_numpy(), "is_churn": p}).to_parquet(
        PRED_DIR / f"test_{mid}.parquet", index=False)

# Submissions: one per model and the primary winner.
for mid in MODEL_IDS:
    pd.DataFrame({"msno": scored_test["msno"].to_numpy(), "is_churn": TEST_PROBA[mid]}).to_csv(
        SUB_DIR / f"submission_{mid}.csv", index=False)
pd.DataFrame({"msno": scored_test["msno"].to_numpy(), "is_churn": TEST_PROBA[primary_winner]}).to_csv(
    SUB_DIR / "submission.csv", index=False)
print("submissions written. primary:", primary_winner)


In [ ]:
# ---- Per-model manifests and a reloadable predictor interface -------------
def _manifest(mid):
    rec = REGISTRY[mid]
    man = {
        "model_id": mid, "family": rec["family"], "input": rec["input"],
        "dataset_version": META["dataset_version"], "contract_hash": CONTRACT_HASH,
        "n_channels": rec.get("n_channels", "N/A"),
        "best_params": rec.get("best_params", "N/A"),
        "threshold": THRESHOLDS[mid], "fit_rate": FINAL_RATE,
        "libs": ENV, "seed": SEED,
        "preprocessing": {"tabular": "preprocessing/final/tabular_preprocess.json"},
    }
    if mid == "M1":
        man["model_file"] = "models/final/M1_logistic.joblib"
    if mid == "M2":
        man["model_file"] = "models/final/M2_lightgbm.txt"
        man["best_iteration"] = rec["best_iteration"]
        man["tree_vocab"] = tree_vocab("train")
    if mid == "M3":
        man["model_file"] = "models/final/M3_xgboost.json"
        man["best_iteration"] = rec["best_iteration"]
        man["tree_vocab"] = tree_vocab("train")
    if mid == "M4":
        man["model_file"] = "models/final/M4_randomforest.joblib"
        man["rf_vocab"] = rf_vocab("train")
    return man

MANIFEST_DIR = OUT_DIR / "models" / "manifests"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
MANIFESTS = {mid: _manifest(mid) for mid in MODEL_IDS}
for mid, man in MANIFESTS.items():
    with (MANIFEST_DIR / f"{mid}.json").open("w") as f:
        json.dump(man, f, indent=2, default=str)
holdout_metrics.to_csv(OUT_DIR / "metrics" / "holdout_metrics.csv", index=False)
with (OUT_DIR / "run_config.json").open("w") as f:
    json.dump({"env": ENV, "smoke": SMOKE_TEST, "trials": TRIALS,
               "contract": CONTRACT, "selection": SELECTION}, f, indent=2, default=str)
print("saved manifests, run_config.json and metrics")


In [ ]:
# ---- Reloadable scoring by model_id (no notebook globals needed) ----------
def load_predictor(model_id):
    """Rebuild a predictor from its manifest + artifacts, independent of run state."""
    man = json.loads((MANIFEST_DIR / f"{model_id}.json").read_text())
    family = man["family"]
    if family == "logistic":
        model = joblib.load(OUT_DIR / man["model_file"])
        return lambda feats, seq=None: model.predict_proba(feats[FEATURES])[:, 1], man
    if family == "lightgbm":
        booster = lgb.Booster(model_file=str(OUT_DIR / man["model_file"]))
        def predict_lgb(feats, seq=None):
            frame = feats[FEATURES].copy()
            for c in CATEGORICAL:
                frame[c] = pd.Categorical(frame[c], categories=man["tree_vocab"][c])
            return booster.predict(frame)
        return predict_lgb, man
    if family == "xgboost":
        model = XGBClassifier(); model.load_model(str(OUT_DIR / man["model_file"]))
        def predict_xgb(feats, seq=None):
            frame = feats[FEATURES].copy()
            for c in CATEGORICAL:
                frame[c] = pd.Categorical(frame[c], categories=man["tree_vocab"][c])
            return model.predict_proba(frame)[:, 1]
        return predict_xgb, man
    if family == "randomforest":
        model = joblib.load(OUT_DIR / man["model_file"])
        def predict_rf(feats, seq=None):
            return model.predict_proba(encode_rf_frame(feats, man["rf_vocab"]))[:, 1]
        return predict_rf, man
    raise ValueError(f"Unknown family {family!r} for the tabular notebook.")


def score_model(model_id, features_df, seq=None):
    """Return msno + probability for one model id, by its manifest. Errors are explicit."""
    predictor, man = load_predictor(model_id)
    proba = predictor(features_df, seq)
    proba = np.clip(np.asarray(proba, dtype=np.float64), EPS, 1 - EPS)
    return pd.DataFrame({"msno": features_df["msno"], "is_churn": proba})


def evaluate_with_targets(model_id, features_df, targets, seq=None):
    """Join a target by unique msno (no silent row loss) and report metrics at saved threshold."""
    preds = score_model(model_id, features_df, seq).rename(columns={"is_churn": "proba"})
    merged = preds.merge(targets[["msno", "is_churn"]], on="msno", how="inner")
    if len(merged) != len(targets):
        raise ValueError(f"target join lost rows: {len(targets)} -> {len(merged)}")
    man = json.loads((MANIFEST_DIR / f"{model_id}.json").read_text())
    return evaluate(merged["is_churn"].to_numpy(), merged["proba"].to_numpy(),
                    model_id, man["threshold"])


In [ ]:
# ---- Reload check: final scoring matches in-memory inference (tolerance) --
_check_id = primary_winner
_rows_check = test_all_rows[:256]
_feats_check = test_df.iloc[_rows_check]
_reloaded = score_model(_check_id, _feats_check)["is_churn"].to_numpy()
_in_mem = np.asarray(TEST_PROBA[_check_id])[_rows_check]
print(f"reload check for {_check_id}: max abs diff =",
      float(np.max(np.abs(_reloaded - _in_mem))))
assert np.allclose(_reloaded, _in_mem, atol=1e-4), "reloaded scoring diverged"
print("reloadable scoring verified within tolerance")


## 13. Notes

**What this notebook does.** Four comparable tabular models (M1 logistic, M2 LightGBM,
M3 XGBoost, M4 RandomForest) share one split (`fit`/`stop`/`tune`/`holdout`). Each trainable
family has its own Optuna study persisted in SQLite (resumable by total budget, rejecting a
changed data/split contract). Metrics, confusion matrices (counts + row-normalised), ROC/PR,
reliability and Optuna-history curves are produced for every model. The winner and thresholds
are fixed on `tune` and written to `selection_manifest.json` **before** holdout is read.
Families are then refit on all development and evaluated once on holdout, and finally refit
on 100% of train for test inference. Reloadable scoring (`load_predictor` / `score_model`)
rebuilds each predictor from its manifest without notebook globals.

**Honest scope of results.**
- `SMOKE_TEST=True` uses a small user sample and few trials. Its numbers are **not** FULL
  results and go to `*_smoke/`. Set `SMOKE_TEST=False` for the full comparison.
- The expiry holdout is a 20% latest-expiry control, not cross-month validation: every
  labelled user belongs to the same competition cutoff.
- Reliability curves are diagnostics; no probability calibration was applied.
- The v3 tabular aggregates keep the competition-train cleaning parameters (inherited
  limitation); this modelling pipeline adds no further holdout/test fitting.

**Artifacts** are written under `OUT_DIR` (`run_config.json`, `split_indices.npz`,
`split_summary.csv`, `selection_manifest.json`, `optuna/`, `preprocessing/`, `models/`,
`metrics/`, `predictions/`, `figures/`, `submissions/`). No submission is uploaded to
Kaggle automatically.

The neural models (3 LSTM + hybrid MLP) and their sequences live in the w2 neural notebook.
